<a href="https://colab.research.google.com/github/julianovale/PO1/blob/main/PO1_001_Problema_da_mistura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ortools

# Problema da Mistura (Ração Animal)

O objetivo é determinar a combinação exata de ingredientes que atenda a todas as necessidades nutricionais exigidas ao menor custo financeiro possível.

A demanda estabelece que **1 kg de ração animal** deve conter, no mínimo:
* 300g de proteína
* 500g de cálcio
* *(Não há limites máximos estabelecidos).*

## Tabela de Ingredientes

| Ingrediente | Proteína (g/kg) | Cálcio (g/kg) | Custo (R$/kg) |
| :--- | :--- | :--- | :--- |
| **Osso** | 200 | 600 | 0,56 |
| **Soja** | 500 | 400 | 0,81 |
| **Peixe** | 400 | 400 | 0,26 |

## Modelo Matemático

**Variáveis de Decisão:**
* $x_1$: Quantidade de Osso (em kg)
* $x_2$: Quantidade de Soja (em kg)
* $x_3$: Quantidade de Peixe (em kg)

**Função Objetivo (Minimizar Custos):**
$\min Z = 0.56x_1 + 0.81x_2 + 0.26x_3$

**Restrições:**
* **Total da mistura:** $x_1 + x_2 + x_3 = 1$
* **Mínimo de Proteína:** $200x_1 + 500x_2 + 400x_3 \ge 300$
* **Mínimo de Cálcio:** $600x_1 + 400x_2 + 400x_3 \ge 500$
* **Não-negatividade:** $x_1, x_2, x_3 \ge 0$

# OR-Tools

In [ ]:
from ortools.linear_solver import pywraplp

def resolver_problema_racao():
    # Instancia o solver GLOP (específico para Programação Linear)
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return

    # 1. Variáveis de Decisão (com limite inferior 0 e superior infinito)
    x1 = solver.NumVar(0, solver.infinity(), 'Osso')
    x2 = solver.NumVar(0, solver.infinity(), 'Soja')
    x3 = solver.NumVar(0, solver.infinity(), 'Peixe')

    # 2. Restrições
    # Restrição de peso: x1 + x2 + x3 = 1 kg
    solver.Add(x1 + x2 + x3 == 1.0, 'Peso_Total')

    # Restrição de proteína: >= 300g
    solver.Add(200 * x1 + 500 * x2 + 400 * x3 >= 300, 'Min_Proteina')

    # Restrição de cálcio: >= 500g
    solver.Add(600 * x1 + 400 * x2 + 400 * x3 >= 500, 'Min_Calcio')

    # 3. Função Objetivo: Minimizar Custos
    # Z = 0.56*x1 + 0.81*x2 + 0.26*x3
    solver.Minimize(0.56 * x1 + 0.81 * x2 + 0.26 * x3)

    # 4. Resolve o problema
    status = solver.Solve()

    # 5. Exibe os resultados
    if status == pywraplp.Solver.OPTIMAL:
        print('Solução Ótima Encontrada!')
        print(f'Custo mínimo para 1 kg de ração: R$ {solver.Objective().Value():.2f}\n')
        print('Receita (Quantidade de cada ingrediente):')
        print(f'- {x1.name()}: {x1.solution_value() * 1000:.0f} gramas')
        print(f'- {x2.name()}: {x2.solution_value() * 1000:.0f} gramas')
        print(f'- {x3.name()}: {x3.solution_value() * 1000:.0f} gramas')

        # Validando os nutrientes finais
        prot_final = 200 * x1.solution_value() + 500 * x2.solution_value() + 400 * x3.solution_value()
        calc_final = 600 * x1.solution_value() + 400 * x2.solution_value() + 400 * x3.solution_value()

        print('\nValores Nutricionais Entregues:')
        print(f'- Proteína: {prot_final:.1f}g')
        print(f'- Cálcio: {calc_final:.1f}g')
    else:
        print('O problema não possui uma solução ótima viável.')

# Executa a função
resolver_problema_racao()

Solução Ótima Encontrada!
Custo mínimo para 1 kg de ração: R$ 0.41

Receita (Quantidade de cada ingrediente):
- Osso: 500 gramas
- Soja: 0 gramas
- Peixe: 500 gramas

Valores Nutricionais Entregues:
- Proteína: 300.0g
- Cálcio: 500.0g


# scipy.optimize.linprog

In [7]:
import numpy as np
from scipy.optimize import linprog

# 1. Vetor de custos da Função Objetivo: min Z = 0.56*x1 + 0.81*x2 + 0.26*x3
c = np.array([0.56, 0.81, 0.26])

# 2. Restrições de desigualdade (formato obrigatório: A_ub @ x <= b_ub)
# Multiplicamos as restrições de >= por -1 para inverter o sinal:
# Proteína: -200*x1 - 500*x2 - 400*x3 <= -300
# Cálcio:   -600*x1 - 400*x2 - 400*x3 <= -500
A_ub = np.array([
    [-200, -500, -400],
    [-600, -400, -400]
])
b_ub = np.array([-300, -500])

# 3. Restrição de igualdade (A_eq @ x == b_eq)
# Peso total: 1*x1 + 1*x2 + 1*x3 = 1
A_eq = np.array([
    [1, 1, 1]
])
b_eq = np.array([1])

# 4. Limites das variáveis (não-negatividade: x1, x2, x3 >= 0)
bounds = [(0, None), (0, None), (0, None)]

# 5. Resolução do problema
resultado = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method='highs'
)

# 6. Exibição dos resultados
if resultado.success:
    x1, x2, x3 = resultado.x
    print('Solução Ótima Encontrada!')
    print(f'Custo mínimo para 1 kg de ração: R$ {resultado.fun:.2f}\n')
    print('Receita (Quantidade de cada ingrediente):')
    print(f'- Osso: {x1 * 1000:.0f} gramas')
    print(f'- Soja: {x2 * 1000:.0f} gramas')
    print(f'- Peixe: {x3 * 1000:.0f} gramas')

    # Validando os nutrientes finais
    prot_final = 200 * x1 + 500 * x2 + 400 * x3
    calc_final = 600 * x1 + 400 * x2 + 400 * x3
    print('\nValores Nutricionais Entregues:')
    print(f'- Proteína: {prot_final:.1f}g')
    print(f'- Cálcio: {calc_final:.1f}g')
else:
    print('Não foi possível encontrar uma solução:', resultado.message)

Solução Ótima Encontrada!
Custo mínimo para 1 kg de ração: R$ 0.41

Receita (Quantidade de cada ingrediente):
- Osso: 500 gramas
- Soja: 0 gramas
- Peixe: 500 gramas

Valores Nutricionais Entregues:
- Proteína: 300.0g
- Cálcio: 500.0g
